# Appendix C.4 — Catchment Data Explorer

Evidence for the claims made about the EA's Catchment Data Explorer (CDE) in Appendix C.

**Sources used**

| source | what it is |
| --- | --- |
| `ttl/catchment.ttl` | the graph extracted from the EA's internal SPARQL endpoint, committed |
| `https://environment.data.gov.uk/catchment-planning/…` | the public CDE, probed and downloaded live |
| `ttl/catchment/ISSUES.md` | defects recorded during that extraction |

Live requests degrade gracefully offline. The internal endpoint the graph came from is **not** publicly
reachable, so C.4.5 is evidenced against the committed extract.


In [1]:
import os, json, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "raw_datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
RAW = ROOT / "raw_datasets"
REG = RAW / "access_database_csv_files"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
print("repository root:", ROOT)


repository root: /Users/waf/git/projects/demonstrator-poc


In [2]:
import urllib.request, urllib.error

CDE = "https://environment.data.gov.uk/catchment-planning"

def probe(url, accept=None):
    """(status, content-type, first bytes) for a URL."""
    req = urllib.request.Request(url, headers={"Accept": accept} if accept else {})
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, r.headers.get("Content-Type", ""), r.read(200)
    except urllib.error.HTTPError as e:
        return e.code, e.headers.get("Content-Type", ""), b""
    except Exception as e:
        return None, str(e), b""

status, ctype, _ = probe(CDE + "/OperationalCatchment/3367")
ONLINE = status is not None
print("network reachable:", ONLINE)


network reachable: True


---
## C.4.1 The RDF is complete and correct; the published CSVs are a lossy view of it

> *"The source graph is well modelled — RDF Data Cube, versioned water bodies, resolvable concept
> schemes, geometry included. None of it is served. The public surface is CSV, and the CSV drops
> records, history and geometry that the graph holds."*

The demonstrator's `ttl/catchment.ttl` was extracted from the EA's internal SPARQL endpoint. This
section compares what that extract holds against what the public CSVs expose, so the cost of the
CSV-only surface is a measured quantity rather than a complaint.


In [3]:
import io
import pyoxigraph as ox

store = ox.Store()
store.bulk_load(path=str(ROOT / "ttl" / "catchment.ttl"), format=ox.RdfFormat.TURTLE)
print(f"ttl/catchment.ttl: {len(store):,} triples")

def sparql(q, cols):
    return pd.DataFrame([[None if v is None else v.value for v in r] for r in store.query(q)], columns=cols)

csv = cls = None
if ONLINE:
    with urllib.request.urlopen(CDE + "/OperationalCatchment/3367/rnags.csv", timeout=60) as r:
        csv = pd.read_csv(io.BytesIO(r.read()))
    with urllib.request.urlopen(CDE + "/OperationalCatchment/3367/classifications.csv", timeout=120) as r:
        cls = pd.read_csv(io.BytesIO(r.read()))
    print(f"rnags.csv: {len(csv)} rows    classifications.csv: {len(cls):,} rows")
else:
    print("offline -- CSV comparisons are skipped; recorded values appear in the prose")


ttl/catchment.ttl: 51,525 triples


rnags.csv: 93 rows    classifications.csv: 5,852 rows


### Why a consumer is on the CSV at all

The application looks like linked data and serves none: RDF `Accept` headers return **200** with an HTML
body, so a content-negotiation probe reports success and then fails to parse a web page. Format suffixes
return 500, `/sparql` 404s, and the `so/` URIs the graph uses as subjects do not dereference.


In [4]:
WB = CDE + "/WaterBody/GB108044009920"          # a real Poole Harbour water body

if ONLINE:
    rows = []
    for label, url, accept in [
        ("Accept: text/turtle",           WB, "text/turtle"),
        ("Accept: application/rdf+xml",   WB, "application/rdf+xml"),
        ("Accept: application/ld+json",   WB, "application/ld+json"),
        ("suffix .ttl",                   WB + ".ttl", None),
        ("suffix .rdf",                   WB + ".rdf", None),
        ("suffix .json",                  WB + ".json", None),
        ("the SPARQL endpoint",           CDE + "/sparql", None),
    ]:
        s, ct, head = probe(url, accept)
        rows.append({"request": label, "status": s, "content-type": ct,
                     "body starts": head[:40].decode("utf-8", "replace")})
    display(pd.DataFrame(rows))
else:
    print("offline -- recorded result:")
    print("  every RDF request above returned 200 text/html; /sparql returned 404")


,request,status,content-type,body starts
0,Accept: text/turtle,200,text/html;charset=utf-8,"<!DOCTYPE html>\n<html lang=""en""><head><m"
1,Accept: application/rdf+xml,200,text/html;charset=utf-8,"<!DOCTYPE html>\n<html lang=""en""><head><m"
2,Accept: application/ld+json,200,text/html;charset=utf-8,"<!DOCTYPE html>\n<html lang=""en""><head><m"
3,suffix .ttl,500,text/html,
4,suffix .rdf,500,text/html,
5,suffix .json,500,text/html,
6,the SPARQL endpoint,404,text/html;charset=utf-8,


In [5]:
if ONLINE:
    for url in [CDE + "/so/WaterBody/GB108044009920", CDE + "/so/OperationalCatchment/3367"]:
        s, ct, _ = probe(url, "text/turtle")
        print(f"{s}  {ct:<32} {url}")
    print("\nThe same water body's human-readable page, without the 'so/':")
    s, ct, _ = probe(WB)
    print(f"{s}  {ct:<32} {WB}")
else:
    print("offline -- recorded result: the so/ URIs 404; the so/-less page returns 200 text/html")


404  text/html;charset=utf-8          https://environment.data.gov.uk/catchment-planning/so/WaterBody/GB108044009920
404  text/html;charset=utf-8          https://environment.data.gov.uk/catchment-planning/so/OperationalCatchment/3367

The same water body's human-readable page, without the 'so/':


200  text/html;charset=utf-8          https://environment.data.gov.uk/catchment-planning/WaterBody/GB108044009920


### What the CSV costs, counted

In [6]:
# What the graph holds, against what the published CSVs carry.
rows = []

def graph_count(query):
    return int(next(iter(store.query(query)))[0].value)

CP = "http://environment.data.gov.uk/catchment-planning/def/"
rows.append({"fact": "classification records",
             "in the RDF": graph_count(f"SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE {{ ?c a <{CP}waterbody-classification/Classification> }}"),
             "in the CSV": len(cls) if cls is not None else None,
             "surface": "classifications.csv"})
rows.append({"fact": "RNAG challenges",
             "in the RDF": graph_count(f"SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE {{ ?c a <{CP}reason-for-failure/ReasonForFailure> }}"),
             "in the CSV": int(csv["ID"].nunique()) if csv is not None else None,
             "surface": "rnags.csv"})
rows.append({"fact": "water-body versions carrying a designation",
             "in the RDF": graph_count(f"SELECT (COUNT(*) AS ?n) WHERE {{ ?v <{CP}water-framework-directive/hydromorphologicalDesignation> ?d }}"),
             "in the CSV": int(csv["Water Body ID"].nunique()) if csv is not None else None,
             "surface": "current version only"})
rows.append({"fact": "geometries (catchment polygon + river line)",
             "in the RDF": graph_count("SELECT (COUNT(DISTINCT ?g) AS ?n) WHERE { ?g <http://www.opengis.net/ont/geosparql#asWKT> ?w }"),
             "in the CSV": 0,
             "surface": "point Easting/Northing only"})
rows.append({"fact": "SKOS concepts with labels and scheme membership",
             "in the CSV": 0,
             "in the RDF": graph_count("SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c a <http://www.w3.org/2004/02/skos/core#Concept> }"),
             "surface": "bare strings"})
pd.DataFrame(rows)[["fact", "in the RDF", "in the CSV", "surface"]]


,fact,in the RDF,in the CSV,surface
0,classification records,5852,5852,classifications.csv
1,RNAG challenges,95,93,rnags.csv
2,water-body versions carrying a designation,39,19,current version only
3,geometries (catchment polygon + river line),38,0,point Easting/Northing only
4,SKOS concepts with labels and scheme membership,158,0,bare strings


Read down the table. The classification series survives export intact — 5,852 records, all ten years,
all three cycles, item-level, with certainty and confidence. That part of the CSV is faithful, and it is
the bulk of the dataset.

What does not survive is everything the CSV's flat shape cannot hold:

- **Records** — two complete cycle-3 RNAGs are missing (C.4.2).
- **History** — the designation attaches to a *versioned* water body; the CSV reports the current
  version, so three rivers reclassified between versions look like they never moved (C.4.4).
- **Geometry** — the graph carries a catchment polygon and a river line per water body as `geo:asWKT`,
  so no GeoJSON merge was needed to build the map. The CSV carries a single centroid-ish point, and the
  boundaries are a separate shapefile download.
- **Vocabulary** — 158 SKOS concepts with labels and scheme membership become bare strings, which is why
  the cross-table's grouping column is ambiguous (C.4.3).


---
## C.4.2 The published CSV is not a faithful export — two RNAGs are missing

> *"Two complete cycle-3 RNAGs present in the source graph are absent from `rnags.csv` (95 vs 93)."*


In [7]:
graph_rnags = sparql("""
SELECT ?rnag WHERE {
  ?rnag a <http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/ReasonForFailure> }""",
  ["rnag"])
graph_ids = set(graph_rnags.rnag.str.rsplit("/", n=1).str[-1])
print(f"RNAGs in the extracted graph: {len(graph_ids)}")
if csv is not None:
    print(f"RNAGs in the published CSV  : {csv['ID'].nunique()}")
    print(f"\nIn the graph, absent from the CSV: {sorted(graph_ids - set(csv['ID'].astype(str)))}")
else:
    print("offline -- recorded result: the published CSV carries 93")


RNAGs in the extracted graph: 95
RNAGs in the published CSV  : 93

In the graph, absent from the CSV: ['578245', '578282']


In [8]:
missing = sorted(graph_ids - set(csv["ID"].astype(str))) if csv is not None else []
if missing:
    for rnag in missing:
        print(f"Everything the graph holds about RNAG {rnag}, which the CSV omits:")
        display(sparql("""
SELECT ?predicate ?object WHERE {
  <http://environment.data.gov.uk/catchment-planning/data/reason-for-failure/%s> ?predicate ?object }
ORDER BY ?predicate""" % rnag, ["predicate", "object"]))
else:
    print("offline -- see ttl/catchment/PLAN.md for the recorded comparison")


Everything the graph holds about RNAG 578245, which the CSV omits:


,predicate,object
0,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/activity,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/activity_1152
1,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/activityCerta...,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/cert_3
2,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/apportionment,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/app_6
3,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/businessSector,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/bcs_107
4,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/category,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/bcs_13
5,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/categoryCerta...,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/cert_3
6,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/nationalSWMIh...,Changes to the natural flow and levels of water
7,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/pressureTier3,Abstraction and flow
8,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/problem,http://environment.data.gov.uk/catchment-planning/data/problem/426861
9,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/reasonForFail...,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/rff


Everything the graph holds about RNAG 578282, which the CSV omits:


,predicate,object
0,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/activity,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/activity_1096
1,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/activityCerta...,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/cert_1
2,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/apportionment,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/app_6
3,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/businessSector,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/bcs_107
4,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/category,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/bcs_7
5,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/categoryCerta...,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/cert_4
6,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/pressureTier3,Abstraction and flow
7,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/problem,http://environment.data.gov.uk/catchment-planning/data/problem/426817
8,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/reasonForFail...,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/rff
9,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/swmi,http://environment.data.gov.uk/catchment-planning/def/reason-for-failure/swmi_43


These are complete records, not fragments: full certainty triples, business sector, pressure and
activity. Nothing marks them as different from the 93 that were exported. A consumer working from the
CSV — the only surface the public site offers — cannot know they exist.


---
## C.4.3 The published cross-table's counting rule is not documented

> *"It is not a row count … It also omits challenges carrying no national heading — 57 of this
> catchment's 95."*

Two populations, counted separately below. **57 of 95** is the graph, which holds two RNAGs the CSV
omits — one of them, `578282`, carries no heading. **56 of 93** is the same count against the published
CSV.


In [9]:
if csv is not None:
    blank = csv["National Swmi Header"].isna() | (csv["National Swmi Header"].astype(str).str.strip() == "")
    print(f"CSV rows with no National SWMI Header: {blank.sum()} of {len(csv)}")
    print("\nThose rows are perfectly real challenges -- here are five:")
    display(csv[blank][["Water Body","Classification Status","Business Sector",
                        "Pressure Tier 3","National Swmi Header"]].head(5))
else:
    print("offline -- recorded result: 56 of 93 CSV rows carry no National SWMI Header")


CSV rows with no National SWMI Header: 56 of 93

Those rows are perfectly real challenges -- here are five:


,Water Body,Classification Status,Business Sector,Pressure Tier 3,National Swmi Header
2,Frome Dorset Trib (River Win),Fail,Not applicable,Chemicals,NaN
4,Frome Dorset Trib (River Win),Fail,Not applicable,Chemicals,NaN
6,Frome Trib (Luckford Lake),Fail,Not applicable,Chemicals,NaN
7,Frome Dorset (Headwaters),Moderate,Not applicable,Phosphate,NaN
8,Frome Trib (Luckford Lake),Fail,Not applicable,Chemicals,NaN


In [10]:
if csv is not None:
    below = csv[csv["Classification Status"].isin(
        ["Bad", "Poor", "Moderate", "Fail", "Does Not Support Good"])]
    print("Plausible readings of 'how many challenges does this catchment have?':\n")
    print(f"  rows in rnags.csv                                          {len(csv)}")
    print(f"  ... restricted to below-good statuses                      {len(below)}")
    print(f"  distinct (water body, status, pressure tier 3)             "
          f"{len(below.drop_duplicates(['Water Body ID','Classification Status','Pressure Tier 3']))}")
    print(f"  distinct water bodies                                      {below['Water Body ID'].nunique()}")
    print(f"  what the published Challenges table shows                  29")


Plausible readings of 'how many challenges does this catchment have?':

  rows in rnags.csv                                          93
  ... restricted to below-good statuses                      90
  distinct (water body, status, pressure tier 3)             46
  distinct water bodies                                      19
  what the published Challenges table shows                  29


The rule that *does* reproduce 29 was reverse-engineered (it is recorded in
`ttl/catchment/PLAN.md` §3). It requires guessing three separate things: which statuses count as below
good, that the count is of **distinct** `(water body, status, pressure tier 3)` triples rather than
rows, and that the grouping is by `Category` — not by `Business Sector`, the other sector-shaped column
sitting right next to it in the same file.


In [11]:
if csv is not None:
    cross = below.groupby(["Category", "National Swmi Header"]).apply(
        lambda g: g.drop_duplicates(["Water Body ID","Classification Status","Pressure Tier 3"]).shape[0],
        include_groups=False)
    print(f"cells: {len(cross)}   total: {cross.sum()}   (the published table: 8 cells, total 29)\n")
    display(cross.rename("challenges").to_frame())


cells: 8   total: 29   (the published table: 8 cells, total 29)



challenges
Category                              National Swmi Header                                       
Agriculture and rural land management Physical modifications                                    2
                                      Pollution from rural areas                               12
Industry                              Pollution from towns, cities and transport                2
Recreation                            Pollution from towns, cities and transport                3
Sector under investigation            Physical modifications                                    1
Urban and transport                   Pollution from towns, cities and transport                1
Water Industry                        Changes to the natural flow and levels of water           1
                                      Pollution from waste water                                7

In [12]:
if csv is not None:
    print("Grouping by the OTHER sector column in the same file gives a different answer:")
    alt = below.groupby(["Business Sector", "National Swmi Header"]).apply(
        lambda g: g.drop_duplicates(["Water Body ID","Classification Status","Pressure Tier 3"]).shape[0],
        include_groups=False)
    print(f"  by Category       -> {len(cross)} cells, total {cross.sum()}")
    print(f"  by Business Sector -> {len(alt)} cells, total {alt.sum()}")
    print("\nAnd what never appears in the published table at all:")
    print(csv.Category.value_counts().to_string())


Grouping by the OTHER sector column in the same file gives a different answer:
  by Category       -> 8 cells, total 29
  by Business Sector -> 11 cells, total 33

And what never appears in the published table at all:
Category
No sector responsible                    55
Agriculture and rural land management    19
Water Industry                           10
Recreation                                3
Industry                                  3
Urban and transport                       1
Sector under investigation                1
Other                                     1


55 of the 93 challenges are attributed to "No sector responsible" and carry no national SWMI header,
so they are absent from the published cross-table entirely — 60% of the catchment's challenges. A reader
of that table cannot tell they were excluded rather than absent.


---
## C.4.4 The CSV reports only the current water-body version, concealing a real change

> *"the natural / artificial / heavily-modified designation attaches to the versioned water body and is
> not constant — three rivers here were heavily modified at v1 and ceased to be so at v2."*


In [13]:
designations = sparql("""
PREFIX wfd: <http://environment.data.gov.uk/catchment-planning/def/water-framework-directive/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?version ?designation ?label WHERE {
  ?version wfd:hydromorphologicalDesignation ?designation .
  OPTIONAL { ?designation rdfs:label ?label }
} ORDER BY ?version""", ["version", "designation", "label"])
designations["water body"] = designations.version.str.extract(r"/WaterBody/([^/]+)/")[0]
designations["v"] = designations.version.str.extract(r"/(\d+)$")[0]
print(f"{len(designations)} version-level designation statements across "
      f"{designations['water body'].nunique()} water bodies")
designations[["water body", "v", "label"]].head(8)


39 version-level designation statements across 19 water bodies


,water body,v,label
0,GB108044009610,1,not designated artificial or heavily modified
1,GB108044009610,2,not designated artificial or heavily modified
2,GB108044009620,1,not designated artificial or heavily modified
3,GB108044009620,2,not designated artificial or heavily modified
4,GB108044009630,1,not designated artificial or heavily modified
5,GB108044009630,2,not designated artificial or heavily modified
6,GB108044009640,1,not designated artificial or heavily modified
7,GB108044009640,2,not designated artificial or heavily modified


In [14]:
changed = designations.groupby("water body").label.nunique()
changed = changed[changed > 1]
print(f"water bodies whose designation is NOT constant across versions: {len(changed)}\n")
designations[designations["water body"].isin(changed.index)][
    ["water body", "v", "label"]].sort_values(["water body", "v"]).reset_index(drop=True)


water bodies whose designation is NOT constant across versions: 3



,water body,v,label
0,GB108044009700,1,heavily modified
1,GB108044009700,2,not designated artificial or heavily modified
2,GB108044009700,3,not designated artificial or heavily modified
3,GB108044009780,1,heavily modified
4,GB108044009780,2,not designated artificial or heavily modified
5,GB108044009780,3,not designated artificial or heavily modified
6,GB108044010080,1,heavily modified
7,GB108044010080,2,not designated artificial or heavily modified
8,GB108044010080,3,not designated artificial or heavily modified


In [15]:
if csv is not None:
    print("What the published CSV says about the same three water bodies -- one value each:")
    display(csv[csv["Water Body ID"].isin(changed.index)][
        ["Water Body","Water Body ID","Hydromorphological designation"]].drop_duplicates()
        .reset_index(drop=True))


What the published CSV says about the same three water bodies -- one value each:


,Water Body,Water Body ID,Hydromorphological designation
0,Sydling Water,GB108044009700,not designated artificial or heavily modified
1,Frome Dorset (Upper),GB108044009780,not designated artificial or heavily modified
2,Piddle (Lower),GB108044010080,not designated artificial or heavily modified


Read from the CSV, the designation scheme looks degenerate — a single value across all 19 water
bodies, useless for analysis. It is degenerate *in the current version only*. The history exists, is
modelled, and is dropped on export.


---
## C.4.5 A source membership defect: Stannon Lake

> *"A twentieth water body, Stannon Lake (Cornwall), is returned by the obvious catchment-membership
> query for operational catchment 3367."*

**This one cannot be reproduced from anything public**, and that is the point. The defect exists in the
internal graph and is invisible on every published surface — the CDE application filters it out, so the
website is right and the graph it is built on is wrong.


In [16]:
print("The delivered extract excludes it deliberately:")
print(sparql("""
PREFIX wfd: <http://environment.data.gov.uk/catchment-planning/def/water-framework-directive/>
SELECT (COUNT(DISTINCT ?wb) AS ?water_bodies) WHERE { ?wb a wfd:WaterBody }""",
["water_bodies"]).to_string(index=False))

if csv is not None:
    print("\nThe published CSV agrees -- Stannon Lake does not appear:")
    print(" water bodies in rnags.csv:", csv["Water Body"].nunique())
    print(" any mention of Stannon:   ", csv.apply(
        lambda r: r.astype(str).str.contains("Stannon", case=False).any(), axis=1).sum(), "rows")

print("\nThe query that returns it against the internal endpoint, recorded in ttl/catchment/ISSUES.md:")
print((ROOT / "ttl" / "catchment" / "ISSUES.md").read_text().split("---")[1][:1200])


The delivered extract excludes it deliberately:
water_bodies
          19

The published CSV agrees -- Stannon Lake does not appear:
 water bodies in rnags.csv: 19
 any mention of Stannon:    0 rows

The query that returns it against the internal endpoint, recorded in ttl/catchment/ISSUES.md:


## 1. Stannon Lake is in two operational catchments — HIGH

`cp:so/WaterBody/GB30846165` ("Stannon Lake") asserts `wfd:inOperationalCatchment` for **both**:

- `cp:so/OperationalCatchment/3065` — **Camel**, Cornwall (correct: Stannon Lake is on Bodmin Moor)
- `cp:so/OperationalCatchment/3367` — **Poole Harbour Rivers**, Dorset (~200 km away)

It is the **only waterbody of 14,864 nationally** that sits in more than one operational catchment:

```sparql
SELECT (COUNT(DISTINCT ?wb) AS ?n) WHERE {
  ?wb wfd:inOperationalCatchment ?a, ?b . FILTER(STR(?a) < STR(?b))
}   # -> 1
```

Both catchments are in the South West river basin district, so an RBD-level sanity check does not catch
it. It carries 27